In [ ]:
import numpy as np
import pandas as pd
from boardgames_recsys.data.filtering import filter_df
from boardgames_recsys.data.matrix import *
from boardgames_recsys.models.collaborative_filtering import *
import seaborn as sns
from boardgames_recsys.evaluation.ratings import *
from copy import *
from sklearn.metrics import root_mean_squared_error

from surprise import NMF
from surprise import Dataset, accuracy
from surprise.model_selection import cross_validate, train_test_split
from surprise.reader import Reader

from sklearn.metrics import root_mean_squared_error, mean_absolute_error
%load_ext autoreload
%autoreload 2

In [ ]:
# import DB et set min_reviews

folder = "database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"{folder}/users.csv", index_col=0)

min_reviews = 10 # change to set one
rev_filter = filter_df(avis_clean, min_reviews)

In [ ]:
users_count = rev_filter.groupby("User id")["Game id"].count().reset_index().rename(columns={"Game id":"Count"})
slices = [ (10, 50), (50, 100), (100, 200), (200, 1700)]
all_rmse, all_mae, users = [], [], []
np.random.seed(1)

for (mn, mx) in slices:
    users_ids = users_count[(users_count["Count"] >= mn) & (users_count["Count"] < mx)].sample(77, replace=False)["User id"]
    
    users.append(users_ids.values)
    rmse_user, mae_user = [], []
    
    for user in users_ids:
        games_user = rev_filter[rev_filter["User id"] == user].sort_values("Game id")
        games = games_user.sample(int(0.2 * games_user.shape[0]))["Game id"]

        reviews = rev_filter[(rev_filter["User id"] != user) | ((rev_filter["User id"] == user) & (~rev_filter["Game id"].isin(games)))]
        print(reviews.shape)
        model = NMF(n_factors=20, random_state=42, biased=False, reg_pu= 0.1, reg_qi= 0.1)
        data = Dataset.load_from_df(reviews[["User id", "Game id", "Rating"]], reader=Reader(rating_scale=(0, 10)))
        trainset = data.build_full_trainset()
        nmf = model.fit(trainset)

        U = np.array(nmf.pu)  # User-feature matrix (W)
        G = np.array(nmf.qi)  # Item-feature matrix (H)

        games_nnmf_index = np.array([trainset.to_inner_iid(g) for g in np.sort(games)])
        user_nnmf_index = trainset.to_inner_uid(user)
    
        pred_ratings = U[user_nnmf_index] @ G[games_nnmf_index].T
        true_ratings = games_user[games_user["Game id"].isin(games)]["Rating"]

        rmse, mae = root_mean_squared_error(true_ratings, pred_ratings), mean_absolute_error(true_ratings, pred_ratings)
        print(rmse, mae)
        rmse_user.append(rmse)
        mae_user.append(mae)


    all_rmse.append(rmse_user)
    all_mae.append(mae_user)

In [ ]:
df = pd.DataFrame(data={"User id":users, "RMSE":all_rmse, "MAE":all_mae})

df['zipped'] = df.apply(lambda row: list(zip(row['User id'], row['RMSE'], row['MAE'])), axis=1)
df_exploded = df.explode('zipped', ignore_index=True)
df_exploded[["User id", "RMSE", "MAE"]] = pd.DataFrame(df_exploded['zipped'].tolist(), index=df_exploded.index)
df_exploded = df_exploded.drop(columns='zipped')
df_exploded["Slices"] = ["(10, 50)"] * 77 + ["(50, 100)"] * 77 + ["(100, 200)"] * 77 +  ["(200, 1700)"] * 77 
df_exploded = df_exploded.melt(id_vars=["User id", "Slices"], value_vars=["RMSE", "MAE"], var_name="Metric", value_name="Value")
df_exploded

In [ ]:
sns.set_theme()
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
sns.boxplot(data=df_exploded, x="Slices", y="Value", hue="Metric", ax=ax, showfliers=False, whis=100)
sns.stripplot(data=df_exploded, x="Slices", y="Value", hue="Metric", ax=ax, jitter=True, dodge=True, palette=["black"] * 2, alpha=0.5)
ax.set_ylim(0, 5.8)
ax.set_title("NNMF evaluation per slices")
ax.set_xlabel("Users slices [(min, max) reviews]")
ax.set_ylabel("Error value")
fig.savefig("images/nnmf_eval.png", dpi=150, format="png")
